# Data Scraping

To forecast DK1 electricity prices, we first need data that describes both the price itself and the conditions that influence it. We collect day-ahead prices, electricity consumption, observed weather, and previous-run weather forecasts. The raw files are saved in `data/` and then transformed before they are passed to the model.

| Data | Why it is needed | Saved file |
|------|------------------|------------|
| Day-ahead prices | Target variable and historical price lags | `data/day_ahead_prices_dk1_raw.csv` |
| Weather actuals | Historical weather base for building forecast-like features | `data/weather_actuals_raw.csv` |
| Weather forecasts | Used to estimate realistic forecast errors by horizon | `data/weather_forecasts_raw.csv` |
| Consumption | Grid/load context used later in the resilience analysis | `data/consumption_dk1_raw.csv` |

All raw datasets come from Energi Data Service and Open-Meteo and can be collected with one call:

In [ ]:
from src.data.data_collection import fetch_all

results = fetch_all(start="2021-01-01", end="2026-04-28", price_area="DK1")

As a quick check, we plot prices and actual weather for the same period. This shows that the scraped time series line up and that weather conditions move in ways that are relevant for electricity prices.

In [ ]:
from src.analysis.forecast_analysis import plot_raw_price_and_weather

fig = plot_raw_price_and_weather(start="2025-01-01", end="2025-01-14")

The weather variables appear useful for predicting DK1 electricity prices, especially because several price spikes occur at the same time as changes in wind, temperature, or solar production. The clearest signal is often wind: when wind speeds are low, prices often rise, while periods with stronger wind tend to coincide with lower prices.

# Processing

Before the scraped data can be used for prediction, there is one important issue: in a real forecast setting we do not know future weather actuals. The model should therefore not be trained on perfect future weather observations. It needs weather variables that look like forecasts.

The challenge is that real previous-run weather forecasts are only available from 2025 onwards. To cover the full modelling period from 2021, we use the available real forecasts to measure typical forecast errors, and then use those errors to simulate forecast-like weather data for the earlier period.

The data processing script estimates weather forecast error distributions, builds a forecast-like weather dataset, joins the electricity price target, and creates issue-time price lags. These lags are based on prices before `issue_time`, so they only use information that would have been known when the forecast was made.

In [ ]:
!python -m src.data.data_processing

The first important output is `weather_error_distributions.csv`. The plot below shows how forecast errors change across horizons. Across variables, errors generally widen as the forecast horizon increases, which is expected because longer-range forecasts are more uncertain.

In [ ]:
from src.data.data_processing import plot_weather_error_distributions

plot_weather_error_distributions()

Using these error distributions, the processing script simulates 120-hour weather forecasts for the full historical period. The dashboard-style plot below shows historical actual weather before the issue time and the simulated 5-day forecast afterwards, with actual weather shown for comparison.

In [ ]:
from src.data.data_processing import plot_weather_forecast_dashboard_style

fig_weather = plot_weather_forecast_dashboard_style(
    issue_time="2026-04-23 00:00",
    ctx_days=7,
    show_actuals=True,
)

The processing step creates two main parquet files. `data/forecast_dataset.parquet` contains the forecast-like weather rows, with one row per `(issue_time, horizon_h)` pair. `data/model_dataset.parquet` is the final model-ready table: it adds the matching price target, issue-time price lags, rolling price history, and wind-direction sine/cosine features. This is the file loaded by the XGBoost model.